(projections)=
# Projection Images

```{toctree}
:maxdepth: 2
```

Suite2p computes several reference images during registration and detection. These images are used for quality assessment and as inputs to Cellpose anatomical segmentation.

## Overview

| Image | Source | Cellpose Mode |
|-------|--------|---------------|
| `meanImg` | Mean of registered movie | `anatomical_only=2` |
| `meanImgE` | Enhanced mean (HP filtered) | `anatomical_only=3` |
| `max_proj` | Max of HP-filtered movie | `anatomical_only=4` |
| `refImg` | Registration template | - |
| `Vcorr` | Pixel correlation map | - |

## Mean Image

```{figure} _images/projections/01_mean_image.png
:alt: Mean Image
:name: proj-fig-mean
:width: 80%

Temporal mean of the registered movie. Used with `anatomical_only=2`.
```

```python
meanImg = registered_movie.mean(axis=0)
```

## Enhanced Mean

```{figure} _images/projections/02_mean_enhanced.png
:alt: Enhanced Mean
:name: proj-fig-enhanced
:width: 80%

Spatial high-pass filtered mean image. Sharpens cell boundaries. Used with `anatomical_only=3` (recommended).
```

```python
from scipy.ndimage import median_filter

def enhanced_mean(mean_img, diameter=12):
    I = mean_img.astype(np.float32)
    d = int(4 * np.ceil(diameter) + 1)
    Imed = median_filter(I, size=d)
    I = I - Imed  # subtract local median
    Idiv = median_filter(np.abs(I), size=d)
    return I / (1e-10 + Idiv)
```

## Max Projection

```{figure} _images/projections/03_max_projection.png
:alt: Max Projection
:name: proj-fig-max
:width: 80%

Maximum projection of the temporally HP-filtered movie. Highlights active regions. Used with `anatomical_only=4`.
```

```python
max_proj = hp_filtered_movie.max(axis=0)
```

## Reference Image

```{figure} _images/projections/04_reference_image.png
:alt: Reference Image
:name: proj-fig-ref
:width: 80%

Registration template built from the first N frames. All frames are aligned to this reference.
```

```python
refImg = build_reference(movie[:ops['nimg_init']])
```

## Correlation Map

```{figure} _images/projections/05_correlation_map.png
:alt: Correlation Map
:name: proj-fig-vcorr
:width: 80%

Local pixel correlation map. High values indicate coordinated neural activity. Used for functional ROI detection.
```

```python
Vcorr = local_correlation_map(hp_filtered_movie)
```

## Spatial High-Pass Filter

The `spatial_hp_cp` parameter applies additional high-pass filtering before Cellpose segmentation.

```{figure} _images/projections/06_spatial_hp_filter.png
:alt: Spatial HP Filter
:name: proj-fig-hp
:width: 100%

Effect of `spatial_hp_cp` values on the max projection. Higher values sharpen cell boundaries but may amplify noise.
```

```python
from scipy.ndimage import gaussian_filter

def apply_hp_filter(img, diameter, spatial_hp_cp):
    sigma = diameter * spatial_hp_cp
    return img - gaussian_filter(img, sigma)
```

## Recommendations

```python
ops = {
    "anatomical_only": 3,      # use enhanced mean
    "spatial_hp_cp": 0,        # usually not needed with meanImgE
    "diameter": 6,             # cell size in pixels
    "cellprob_threshold": 0.0,
    "flow_threshold": 0.4,
}
```

If detection is poor:
- Try `anatomical_only=4` (max projection)
- Increase `spatial_hp_cp` to 1-3
- Adjust `diameter` to match cell sizes

## See Also

- {doc}`User Guide <user_guide>` - Complete parameter reference
- [Cellpose Documentation](https://cellpose.readthedocs.io/) - Cellpose model details